# Permutation Test
Works on **any dataset** (CSV / Excel). A file explorer opens, then you choose the columns.

**Idea:** If the null hypothesis $H_0$ (no difference between groups) is true, group labels are *exchangeable*. So:
1. Compute the observed test statistic (e.g. difference in means).
2. Shuffle the group labels many times and recompute the statistic each time. This builds the **null distribution**.
3. **p-value** = fraction of shuffled statistics at least as extreme as the observed one.

No normality assumption is needed.

| Part | Test |
|---|---|
| A | Two independent groups: difference in means / medians / any statistic |
| B | Paired samples (sign-flip permutation) |
| C | Correlation between two numeric variables |
| D | More than two groups (permutation ANOVA) |

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy import stats
rng = np.random.default_rng(42)

## 2. Select Dataset from File Explorer

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)
file_path = filedialog.askopenfilename(
    title="Select your dataset",
    filetypes=[("Data files", "*.csv *.xlsx *.xls *.txt *.tsv"), ("All files", "*.*")]
)
root.destroy()

if not file_path:
    raise SystemExit("No file selected. Re-run this cell.")

print("Selected file:", file_path)

if file_path.endswith('.csv'):
    df = pd.read_csv(file_path)
elif file_path.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_path)
else:
    df = pd.read_csv(file_path, sep=None, engine='python')

print("Shape:", df.shape)
df.head()

## 3. Reusable Permutation Test Function

In [ ]:
def permutation_test(a, b, stat_func=lambda x, y: np.mean(x) - np.mean(y),
                     n_perm=10000, alternative='two-sided', seed=42):
    """
    Two-sample permutation test.
    alternative: 'two-sided', 'greater' (a > b), 'less' (a < b)
    Returns observed statistic, p-value, array of permuted statistics.
    """
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a, float), np.asarray(b, float)
    pooled = np.concatenate([a, b])
    na = len(a)

    observed = stat_func(a, b)
    perm_stats = np.empty(n_perm)
    for i in range(n_perm):
        rng.shuffle(pooled)                      # shuffle labels
        perm_stats[i] = stat_func(pooled[:na], pooled[na:])

    if alternative == 'two-sided':
        count = np.sum(np.abs(perm_stats) >= abs(observed))
    elif alternative == 'greater':
        count = np.sum(perm_stats >= observed)
    else:
        count = np.sum(perm_stats <= observed)

    p_value = (count + 1) / (n_perm + 1)        # +1 correction: p is never exactly 0
    return observed, p_value, perm_stats

# Part A: Two Independent Groups

## 4. Choose the Numeric Variable and the Group Column

In [ ]:
print("Columns:")
for i, c in enumerate(df.columns):
    print(f"{i}: {c}  ({df[c].dtype}, {df[c].nunique()} unique)")

def pick(prompt):
    v = input(prompt).strip()
    return df.columns[int(v)] if v.isdigit() and v not in df.columns else v

value_col = pick("\nNUMERIC variable to compare (name or index): ")
group_col = pick("GROUP column (name or index): ")

data = df[[value_col, group_col]].dropna()
data[value_col] = pd.to_numeric(data[value_col], errors='coerce')
data = data.dropna()

groups = data[group_col].unique()
print("\nGroups found:", list(groups))

In [ ]:
if len(groups) == 2:
    g1, g2 = groups[0], groups[1]
else:
    g1 = input("Enter FIRST group name: ").strip()
    g2 = input("Enter SECOND group name: ").strip()
    g1 = [g for g in groups if str(g) == g1][0]
    g2 = [g for g in groups if str(g) == g2][0]

a = data.loc[data[group_col] == g1, value_col].values
b = data.loc[data[group_col] == g2, value_col].values

print(f"Group '{g1}': n={len(a)}, mean={a.mean():.4f}, median={np.median(a):.4f}, std={a.std(ddof=1):.4f}")
print(f"Group '{g2}': n={len(b)}, mean={b.mean():.4f}, median={np.median(b):.4f}, std={b.std(ddof=1):.4f}")
print(f"Observed difference in means = {a.mean() - b.mean():.4f}")

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
sns.boxplot(x=data[group_col].astype(str), y=data[value_col])
plt.title(f'{value_col} by {group_col}')
plt.subplot(1, 2, 2)
sns.histplot(a, kde=True, color='C0', label=str(g1), stat='density', alpha=.4)
sns.histplot(b, kde=True, color='C1', label=str(g2), stat='density', alpha=.4)
plt.legend(); plt.tight_layout(); plt.show()

## 5. Hypotheses and Settings
$H_0$: the two groups come from the same distribution (no difference in the statistic)  
$H_1$: they differ (two-sided) or one is larger (one-sided)

In [ ]:
alpha = 0.05
n_perm = 10000
alt = 'two-sided'        # change to 'greater' or 'less' for a one-sided test

obs, p_val, perm_diffs = permutation_test(a, b, n_perm=n_perm, alternative=alt)

print(f"Test statistic: difference in means ({g1} - {g2})")
print(f"Observed statistic = {obs:.4f}")
print(f"Permutations       = {n_perm}")
print(f"Alternative        = {alt}")
print(f"p-value            = {p_val:.4f}")
print("Decision:", "REJECT H0" if p_val < alpha else "FAIL TO REJECT H0", f"(alpha = {alpha})")

## 6. Null Distribution Plot

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(perm_diffs, bins=60, color='lightgray', edgecolor='white')
plt.axvline(obs, color='r', lw=2, label=f'Observed = {obs:.3f}')
if alt == 'two-sided':
    plt.axvline(-obs, color='r', lw=2, ls='--', label=f'-Observed')
    tail = np.abs(perm_diffs) >= abs(obs)
else:
    tail = perm_diffs >= obs if alt == 'greater' else perm_diffs <= obs
plt.title(f'Permutation null distribution  (p = {p_val:.4f}, {tail.sum()} of {n_perm} as extreme)')
plt.xlabel('Difference in means under H0'); plt.legend(); plt.show()

## 7. Other Test Statistics (the permutation test works with any statistic)

In [ ]:
stat_funcs = {
    'Difference in means':    lambda x, y: np.mean(x) - np.mean(y),
    'Difference in medians':  lambda x, y: np.median(x) - np.median(y),
    'Ratio of variances':     lambda x, y: np.var(x, ddof=1) / np.var(y, ddof=1),
    'Difference in std dev':  lambda x, y: np.std(x, ddof=1) - np.std(y, ddof=1),
    'Difference in trimmed mean (10%)': lambda x, y: stats.trim_mean(x, .1) - stats.trim_mean(y, .1),
}

rows = []
fig, axes = plt.subplots(1, len(stat_funcs), figsize=(4.2*len(stat_funcs), 3.5))
for ax, (name, f) in zip(axes, stat_funcs.items()):
    o, p, ps = permutation_test(a, b, stat_func=f, n_perm=5000)
    rows.append({'Statistic': name, 'Observed': o, 'p-value': p, 'Reject H0': p < alpha})
    ax.hist(ps, bins=40, color='lightgray'); ax.axvline(o, color='r')
    ax.set_title(name, fontsize=9)
plt.tight_layout(); plt.show()
pd.DataFrame(rows).set_index('Statistic').round(4)

## 8. Compare with Classical Tests

In [ ]:
t_stat, t_p = stats.ttest_ind(a, b, equal_var=False)           # Welch t-test
u_stat, u_p = stats.mannwhitneyu(a, b, alternative='two-sided')

# SciPy's built-in permutation test
res = stats.permutation_test((a, b), lambda x, y: np.mean(x) - np.mean(y),
                             permutation_type='independent', n_resamples=10000,
                             alternative='two-sided', random_state=42)

comp = pd.DataFrame({
    'Test': ['Permutation (ours)', 'Permutation (scipy)', 'Welch t-test', 'Mann-Whitney U'],
    'p-value': [p_val, res.pvalue, t_p, u_p]
}).set_index('Test')
comp['Reject H0'] = comp['p-value'] < alpha
display(comp.round(4))
print("Normality (Shapiro) p-values -> group 1: %.4f, group 2: %.4f"
      % (stats.shapiro(a[:5000])[1], stats.shapiro(b[:5000])[1]))

## 9. Confidence Interval for the Difference (bootstrap)

In [ ]:
boot = np.array([rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(10000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Observed difference = {obs:.4f}")
print(f"95% bootstrap CI    = ({lo:.4f}, {hi:.4f})")
print("CI contains 0 ->", lo <= 0 <= hi, "(consistent with failing to reject H0)" if lo <= 0 <= hi else "(consistent with rejecting H0)")

# Effect size
pooled_sd = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
print(f"Cohen's d           = {obs / pooled_sd:.4f}")

## 10. Does the p-value Stabilise as the Number of Permutations Grows?

In [ ]:
ns = [100, 500, 1000, 5000, 10000, 20000]
pvals = [permutation_test(a, b, n_perm=m, seed=7)[1] for m in ns]
plt.figure(figsize=(7, 4))
plt.plot(ns, pvals, 'o-'); plt.xscale('log')
plt.xlabel('Number of permutations'); plt.ylabel('p-value')
plt.title('p-value estimate vs number of permutations'); plt.grid(alpha=.3); plt.show()

# Part B: Paired Samples (sign-flip permutation test)

Use when the two columns are measurements on the **same subjects** (before/after). Under $H_0$ the sign of each difference is equally likely + or −, so we randomly flip signs.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", num_cols)
run_paired = input("Run paired test? (y/n): ").strip().lower() == 'y'

if run_paired:
    c1 = pick("BEFORE column (name or index): ")
    c2 = pick("AFTER column (name or index): ")
    pair = df[[c1, c2]].dropna().astype(float)
    d = (pair[c1] - pair[c2]).values
    obs_p = d.mean()

    signs = rng.choice([-1, 1], size=(10000, len(d)))
    perm_means = (signs * d).mean(axis=1)
    p_paired = (np.sum(np.abs(perm_means) >= abs(obs_p)) + 1) / (10000 + 1)

    print(f"Mean difference ({c1} - {c2}) = {obs_p:.4f}")
    print(f"Paired permutation p-value    = {p_paired:.4f}")
    print(f"Paired t-test p-value         = {stats.ttest_rel(pair[c1], pair[c2])[1]:.4f}")
    print(f"Wilcoxon signed-rank p-value  = {stats.wilcoxon(pair[c1], pair[c2])[1]:.4f}")

    plt.figure(figsize=(8, 4))
    sns.histplot(perm_means, bins=50, color='lightgray')
    plt.axvline(obs_p, color='r', lw=2); plt.axvline(-obs_p, color='r', ls='--')
    plt.title(f'Sign-flip null distribution (p = {p_paired:.4f})'); plt.show()

# Part C: Correlation Permutation Test

Shuffle one variable to break any relationship, then recompute the correlation.

In [ ]:
run_corr = input("Run correlation test? (y/n): ").strip().lower() == 'y'

if run_corr:
    cx = pick("X column (name or index): ")
    cy = pick("Y column (name or index): ")
    xy = df[[cx, cy]].dropna().astype(float)
    xv, yv = xy[cx].values, xy[cy].values

    r_obs = np.corrcoef(xv, yv)[0, 1]
    perm_r = np.array([np.corrcoef(xv, rng.permutation(yv))[0, 1] for _ in range(10000)])
    p_corr = (np.sum(np.abs(perm_r) >= abs(r_obs)) + 1) / (10000 + 1)

    print(f"Observed Pearson r = {r_obs:.4f}")
    print(f"Permutation p-value = {p_corr:.4f}")
    print(f"Pearson test p-value = {stats.pearsonr(xv, yv)[1]:.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    sns.regplot(x=xv, y=yv, ax=ax[0], scatter_kws={'alpha': .5}); ax[0].set_xlabel(cx); ax[0].set_ylabel(cy)
    sns.histplot(perm_r, bins=50, ax=ax[1], color='lightgray')
    ax[1].axvline(r_obs, color='r', lw=2); ax[1].axvline(-r_obs, color='r', ls='--')
    ax[1].set_title(f'Null distribution of r (p = {p_corr:.4f})')
    plt.tight_layout(); plt.show()

# Part D: More than Two Groups (Permutation ANOVA)

Statistic: F-statistic. Shuffle group labels across all observations.

In [ ]:
run_anova = input("Run multi-group test? (y/n): ").strip().lower() == 'y'

if run_anova:
    vcol = pick("NUMERIC variable (name or index): ")
    gcol = pick("GROUP column with 3+ groups (name or index): ")
    dd = df[[vcol, gcol]].dropna()
    dd[vcol] = pd.to_numeric(dd[vcol], errors='coerce'); dd = dd.dropna()

    labels = dd[gcol].values
    vals = dd[vcol].values
    levels = np.unique(labels)

    def f_stat(v, lab):
        return stats.f_oneway(*[v[lab == l] for l in levels])[0]

    F_obs = f_stat(vals, labels)
    perm_F = np.array([f_stat(vals, rng.permutation(labels)) for _ in range(5000)])
    p_anova = (np.sum(perm_F >= F_obs) + 1) / (5000 + 1)

    print(f"Groups: {list(levels)}")
    print(f"Observed F = {F_obs:.4f}")
    print(f"Permutation p-value = {p_anova:.4f}")
    print(f"Classical ANOVA p-value = {stats.f_oneway(*[vals[labels == l] for l in levels])[1]:.4f}")
    print(f"Kruskal-Wallis p-value  = {stats.kruskal(*[vals[labels == l] for l in levels])[1]:.4f}")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    sns.boxplot(x=dd[gcol].astype(str), y=dd[vcol], ax=ax[0])
    sns.histplot(perm_F, bins=50, ax=ax[1], color='lightgray')
    ax[1].axvline(F_obs, color='r', lw=2); ax[1].set_title(f'Null distribution of F (p = {p_anova:.4f})')
    plt.tight_layout(); plt.show()

# Part E: Type I Error and Power by Simulation (bonus)

In [ ]:
def sim_rejection(effect, n=30, sims=300, n_perm=500):
    rej = 0
    for _ in range(sims):
        x1 = rng.normal(0, 1, n)
        x2 = rng.normal(effect, 1, n)
        _, p, _ = permutation_test(x1, x2, n_perm=n_perm, seed=int(rng.integers(1e9)))
        rej += p < 0.05
    return rej / sims

print("Type I error (no true difference) :", sim_rejection(0))
effects = [0, 0.2, 0.5, 0.8, 1.2]
power = [sim_rejection(e) for e in effects]
plt.figure(figsize=(7, 4))
plt.plot(effects, power, 'o-'); plt.axhline(0.05, color='r', ls='--', label='alpha = 0.05')
plt.xlabel('True mean difference (in SD units)'); plt.ylabel('Rejection rate (power)')
plt.title('Power of the permutation test (n = 30 per group)'); plt.legend(); plt.grid(alpha=.3); plt.show()

## Key Points (for viva)
- **Null hypothesis:** the group labels are exchangeable (no effect). The test is **distribution-free**: no normality or equal-variance assumption.
- **Procedure:** compute observed statistic, shuffle labels many times, build the null distribution, p = proportion of permuted statistics at least as extreme as the observed.
- **p-value formula:** $p = \dfrac{\#\{|T^*| \ge |T_{obs}|\} + 1}{B + 1}$ (the +1 counts the observed arrangement; p is never 0).
- **Exact test** uses all possible permutations ($\binom{n_1+n_2}{n_1}$). **Monte Carlo (approximate) test** uses a random sample of B permutations. The number of permutations does not change the answer much once large (about 10,000).
- **Any statistic can be used:** mean, median, variance, correlation, F.
- **Paired data:** flip signs of differences, not group labels.
- **Permutation test vs bootstrap:** permutation tests **hypothesis** (resampling *without* replacement under H0); bootstrap estimates **confidence intervals** (resampling *with* replacement).
- **Assumption:** exchangeability under H0 (observations independent). With unequal variances, a difference-in-means permutation test can have inflated Type I error.
- **When to use:** small samples, non-normal data, unusual statistics where no classical formula exists.